In [ ]:
# 📌 Install required libraries (Ensures the correct versions are installed)
!pip uninstall -y scikit-learn catboost
!pip install --upgrade --no-cache-dir scikit-learn catboost seaborn

# 📌 Import required libraries
import pandas as pd
import numpy as np
import catboost as cb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 📌 Load dataset
file_path = "games_merged_corrected.csv"
df = pd.read_csv(file_path)

# 📌 Load Hybrid OVR Data
files = {
    "East": "East_Teams_with_Hybrid_OVR.csv",
    "South": "South_Teams_with_Hybrid_OVR_success.csv",
    "North": "North_Teams_with_Hybrid_OVR_success.csv",
    "West": "West_Teams_with_Hybrid_OVR_success.csv"
}

hybrid_ovr_df = pd.concat([pd.read_csv(files[region]) for region in files])
hybrid_ovr_df.rename(columns={"team": "Team_Name", "Hybrid OVR": "Hybrid_OVR"}, inplace=True)

# 📌 Merge Hybrid OVR into dataset
df = df.merge(hybrid_ovr_df, left_on="Team_A", right_on="Team_Name", how="left").rename(columns={"Hybrid_OVR": "Hybrid_OVR_A"}).drop(columns=["Team_Name"])
df = df.merge(hybrid_ovr_df, left_on="Team_B", right_on="Team_Name", how="left").rename(columns={"Hybrid_OVR": "Hybrid_OVR_B"}).drop(columns=["Team_Name"])

# 📌 Fill missing Hybrid OVR values with mean
df["Hybrid_OVR_A"].fillna(df["Hybrid_OVR_A"].mean(), inplace=True)
df["Hybrid_OVR_B"].fillna(df["Hybrid_OVR_B"].mean(), inplace=True)

# 📌 Function to clean dataset
def cleaning(df):
    df_cleaned = df.drop_duplicates()
    for column in df_cleaned.columns:
        if df_cleaned[column].dtype in ['int64', 'float64']:
            df_cleaned[column].fillna(df_cleaned[column].mean(), inplace=True)
        elif df_cleaned[column].dtype == 'object':
            df_cleaned[column].fillna(df_cleaned[column].mode()[0] if not df_cleaned[column].mode().empty else "Unknown", inplace=True)
    return df_cleaned

df = cleaning(df)

# 📌 Create "Winner" column (1 if Team_A wins, 0 if Team_B wins)
df['Winner'] = np.where(df['Score_A'] > df['Score_B'], 1, 0)

# 📌 Select relevant features (Including Hybrid OVR)
features = [
    'Home_A', 'Home_B',
    'FGA_2_A', 'FGA_2_B', 'FGM_2_A', 'FGM_2_B',
    'FGA_3_A', 'FGA_3_B', 'FGM_3_A', 'FGM_3_B',
    'AST_A', 'AST_B', 'BLK_A', 'BLK_B', 'STL_A', 'STL_B',
    'TOV_A', 'TOV_B', 'FTA_A', 'FTA_B', 'FTM_A', 'FTM_B',
    'TOV_team_A', 'TOV_team_B', 'DREB_A', 'DREB_B', 'OREB_A', 'OREB_B',
    'F_tech_A', 'F_tech_B', 'F_personal_A', 'F_personal_B',
    'largest_lead_A', 'largest_lead_B', 'rest_days_A', 'rest_days_B',
    'tz_dif_H_E_A', 'tz_dif_H_E_B', 'prev_game_dist_A', 'prev_game_dist_B',
    'home_away_NS_A', 'home_away_NS_B', 'travel_dist_A', 'travel_dist_B',
    'Hybrid_OVR_A', 'Hybrid_OVR_B'  # ✅ Hybrid OVR included
]

X = df[features]
y = df['Winner']

# 📌 Split data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 📌 Define hyperparameter search space
param_dist = {
    'model__depth': [4, 6, 8],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__l2_leaf_reg': [1, 3, 5, 7, 10],
    'model__iterations': [100, 200, 500],
    'model__bagging_temperature': [0, 0.5, 1]
}

# 📌 Create a pipeline with StandardScaler and CatBoost
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', cb.CatBoostClassifier(
        loss_function='Logloss',
        eval_metric='AUC',
        verbose=0
    ))
])

# 📌 Run RandomizedSearchCV for hyperparameter tuning
random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    scoring='accuracy',
    cv=5,
    n_iter=10,
    random_state=42,
    n_jobs=-1
)

# 📌 Train the model
random_search.fit(X_train, y_train)

# 📌 Get best hyperparameters
best_params = random_search.best_params_
print(f"Best Parameters: {best_params}")

# 📌 Train final model with best hyperparameters
best_model = cb.CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    verbose=0,
    depth=best_params['model__depth'],
    learning_rate=best_params['model__learning_rate'],
    l2_leaf_reg=best_params['model__l2_leaf_reg'],
    iterations=best_params['model__iterations'],
    bagging_temperature=best_params['model__bagging_temperature']
)
best_model.fit(X_train, y_train)

# 📌 Evaluate model
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
logloss = log_loss(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Model Accuracy: {accuracy:.2%}")
print(f"Log Loss: {logloss:.4f}")
print(f"ROC AUC Score: {roc_auc:.4f}")

# 📌 Feature Importance Visualization
feature_importance = best_model.get_feature_importance()
importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': feature_importance})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importance_df['Importance'], y=importance_df['Feature'], palette='viridis')
plt.xlabel("Feature Importance Score")
plt.ylabel("Features")
plt.title("CatBoost Feature Importance")
plt.show()

# 📌 Function to Predict Win Probability (Includes Hybrid OVR)
def predict_winner(team_A, team_B, df, model):
    team_A_ovr = hybrid_ovr_df.loc[hybrid_ovr_df["Team_Name"] == team_A, "Hybrid_OVR"].values
    team_B_ovr = hybrid_ovr_df.loc[hybrid_ovr_df["Team_Name"] == team_B, "Hybrid_OVR"].values

    if len(team_A_ovr) == 0 or len(team_B_ovr) == 0:
        print("Error: One or both teams not found in Hybrid OVR dataset.")
        return

    input_data = pd.DataFrame(columns=X_train.columns)
    input_data.loc[0] = [0] * len(X_train.columns)
    input_data["Hybrid_OVR_A"] = team_A_ovr[0]
    input_data["Hybrid_OVR_B"] = team_B_ovr[0]

    prob = model.predict_proba(input_data)[0][1]
    print(f"Win Probability:\n{team_A}: {prob:.2%} | {team_B}: {(1 - prob):.2%}")

# ✅ Example Prediction
#predict_winner("uconn_huskies", "campbell_fighting_camels", df, best_model)
teams = [
    ("american_university_eagles", "columbia_lions"),
    ("uconn_huskies", "campbell_fighting_camels"),
    ("fairfield_stags", "towson_tigers"),
    ("buffalo_bulls", "stony_brook_seawolves"),
    ("massachusetts_minutewomen", "princeton_tigers"),
    ("drexel_dragons", "delaware_blue_hens"),
    ("liberty_flames", "bucknell_bison"),
    ("nc_state_wolfpack", "north_carolina_tar_heels"),
    ("nc_state_wolfpack", "rhode_island_rams"),
    ("rhode_island_rams", "north_carolina_tar_heels")
]

for team_A, team_B in teams:
    predict_winner(team_A, team_B, df, best_model)